In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2tpdmFuL3JlcG9zL2twb2xpbWlzLmdpdGh1Yi5pby9wb3N0cy9ibG9nL3dvcmxkLWN1cC0yMDI2LWVwbA=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/opt/anaconda3/envs/blog/lib/python3.11/importlib/_bootstrap.py": 1761173985.799012, "/opt/anaconda3/envs/blog/lib/python3.11/importlib/_bootstrap_external.py": 1761173985.7927878, "/opt/anaconda3/envs/blog/lib/python3.11/zipimport.py": 1761173983.5879967, "/opt/anaconda3/envs/blog/lib/python3.11/codecs.py": 1761173982.7493458, "/opt/anaconda3/envs/blog/lib/python3.11/encodings/aliases.py": 1761173984.5492625, "/opt/anaconda3/envs/blog/lib/python3.11/encodings/__init__.py": 1761173984.5448382, "/opt/anaconda3/envs/blog/lib/python3.11/encodings/utf_8.py": 1761173985.17765, "/opt/anaconda3/envs/blog/lib/python3.11/abc.py": 1761173982.6175346, "/opt/anaconda3/envs/blog/lib/python3.11/io.py": 1761173983.0310478, "/opt/anaconda3/envs/blog/lib/python3.11/stat.py": 1761173983.34824, "/opt/anaconda3/envs/blog/lib/python3.11/_collections_abc.py": 1761173982.5577524, "/opt/anaconda3/envs/blog/lib/python3.11/genericpath.py": 1761173982.9408987, "/opt/anaconda3/envs/blog/lib/python3.11/posixpath

In [2]:
#| label: setup
#| include: false

import json
import os

import altair as alt
import numpy as np
import pandas as pd
import footy
from scipy import stats as scipy_stats

# ── Data paths ────────────────────────────────────────────────────────────────
WC_DATA  = "../world-cup-2026-leagues/data"
EPL_DATA = "../../../../epl-standings/data/standings_verified.json"

for f in ["wc_players.csv", "wc_leagues.csv"]:
    if not os.path.exists(f"{WC_DATA}/{f}"):
        raise FileNotFoundError(f"Missing {WC_DATA}/{f} — run fetch_wc_data.py")
if not os.path.exists(EPL_DATA):
    raise FileNotFoundError(f"Missing {EPL_DATA}")

# ── footy helpers ─────────────────────────────────────────────────────────────
def _colour(name: str) -> str:
    try:
        return footy.colour(footy.normalise(name))
    except KeyError:
        return "#AAAAAA"

def _logo(name: str) -> str:
    try:
        return footy.logo(footy.normalise(name)) or ""
    except KeyError:
        return ""

# ── Load WC player data ───────────────────────────────────────────────────────
players = pd.read_csv(f"{WC_DATA}/wc_players.csv")
epl_players = players[players["league"] == "EPL"].copy()
epl_players["club"] = epl_players["club"].str.replace(r"^\d+\. ", "", regex=True).str.strip()

# ── Load EPL standings (2025/26) ──────────────────────────────────────────────
with open(EPL_DATA) as f:
    standings_raw = json.load(f)

latest_season = list(standings_raw.keys())[-1]
final_table = (
    pd.DataFrame.from_dict(standings_raw[latest_season], orient="index",
                           columns=["position"])
    .reset_index()
    .rename(columns={"index": "club"})
    .sort_values("position")
)

# ── Club aggregation ──────────────────────────────────────────────────────────
club_stats = (
    epl_players
    .groupby("club")
    .agg(
        wc_minutes  =("minutes",  "sum"),
        wc_goals    =("goals",    "sum"),
        wc_assists  =("assists",  "sum"),
        player_count=("nation",   "count"),
    )
    .reset_index()
    .sort_values("wc_minutes", ascending=False)
)
nineties = club_stats["wc_minutes"] / 90
club_stats["ga_per90"] = ((club_stats["wc_goals"] + club_stats["wc_assists"]) / nineties).round(2)
club_stats["colour"]   = club_stats["club"].apply(_colour)
club_stats["logo"]     = club_stats["club"].apply(_logo)

# ── Merge EPL table position ──────────────────────────────────────────────────
final_table["club_lower"] = final_table["club"].str.lower().str.strip()
club_stats["club_lower"]  = club_stats["club"].str.lower().str.strip()

# FBref uses short names; EPL table uses full names — map before merge
FBREF_TO_EPL = {
    "manchester utd":   "manchester united",
    "brighton":         "brighton & hove albion",
    "nottingham":       "nottingham forest",
    "newcastle":        "newcastle united",
    "wolves":           "wolverhampton wanderers",
    "tottenham":        "tottenham hotspur",
    "west ham":         "west ham united",
}
club_stats["club_lower"] = club_stats["club_lower"].map(FBREF_TO_EPL).fillna(
    club_stats["club_lower"]
)

merged = club_stats.merge(
    final_table[["club_lower", "position", "club"]],
    on="club_lower", how="left",
    suffixes=("_fbref", "_epl"),
)
merged["display_name"] = merged["club_epl"].fillna(merged["club_fbref"])
merged["logo"]         = merged["club_fbref"].apply(_logo)
merged["colour"]       = merged["club_fbref"].apply(_colour)

# ── Nation breakdown within each club ────────────────────────────────────────
NATION_FLAGS = {
    "England":     "🏴󠁧󠁢󠁥󠁮󠁧󠁿",
    "Argentina":   "🇦🇷",
    "Netherlands": "🇳🇱",
    "Norway":      "🇳🇴",
    "Brazil":      "🇧🇷",
    "Belgium":     "🇧🇪",
    "Spain":       "🇪🇸",
    "Sweden":      "🇸🇪",
    "France":      "🇫🇷",
    "Congo DR":    "🇨🇩",
    "Morocco":     "🇲🇦",
    "Scotland":    "🏴󠁧󠁢󠁳󠁣󠁴󠁿",
    "Other":       "🌍",
}
TOP_NATIONS = 12   # cutoff ≈ 1,450 min (Scotland); nations below → "Other"
TOP_CLUBS   = 15

nation_by_club = (
    epl_players
    .groupby(["club", "nation"])
    .agg(minutes=("minutes", "sum"), player_count=("nation", "count"))
    .reset_index()
)
club_name_map   = merged.set_index("club_fbref")["display_name"].to_dict()
club_colour_map = merged.set_index("club_fbref")["colour"].to_dict()
nation_by_club["display_name"] = nation_by_club["club"].map(club_name_map).fillna(nation_by_club["club"])
nation_by_club["club_colour"]  = nation_by_club["club"].map(club_colour_map).fillna("#AAAAAA")

top_club_list    = club_stats.nlargest(TOP_CLUBS, "wc_minutes")["club"].tolist()
top_club_display = [club_name_map.get(c, c) for c in top_club_list]
top_nations      = set(
    epl_players.groupby("nation")["minutes"].sum()
    .sort_values(ascending=False).head(TOP_NATIONS).index
)

nbc_filtered = nation_by_club[nation_by_club["club"].isin(top_club_list)].copy()
nbc_filtered["nation_label"] = nbc_filtered["nation"].where(
    nbc_filtered["nation"].isin(top_nations), "Other"
)
MIN_SEGMENT = 90   # per-club nation segments below this are folded into "Other"

nbc_agg_raw = (
    nbc_filtered
    .groupby(["display_name", "nation_label"])
    .agg(minutes=("minutes", "sum"), player_count=("player_count", "sum"))
    .reset_index()
)
nbc_agg_raw.loc[nbc_agg_raw["minutes"] < MIN_SEGMENT, "nation_label"] = "Other"
nbc_agg = (
    nbc_agg_raw
    .groupby(["display_name", "nation_label"])
    .agg(minutes=("minutes", "sum"), player_count=("player_count", "sum"))
    .reset_index()
)
nbc_agg["nation_flag"] = nbc_agg["nation_label"].map(NATION_FLAGS).fillna("🌐") + " " + nbc_agg["nation_label"]
nbc_agg["flag_only"]  = nbc_agg["nation_label"].map(NATION_FLAGS).fillna("🌐")

# Cumulative x positions so flag emoji can be centred inside each stacked segment
nbc_sorted = (
    nbc_agg
    .sort_values(["display_name", "minutes"], ascending=[True, False])
    .copy()
)
nbc_sorted["x_end"]   = nbc_sorted.groupby("display_name")["minutes"].cumsum()
nbc_sorted["x_start"] = nbc_sorted["x_end"] - nbc_sorted["minutes"]
nbc_sorted["x_mid"]   = (nbc_sorted["x_start"] + nbc_sorted["x_end"]) / 2

# ── Player detail ─────────────────────────────────────────────────────────────
player_detail = (
    epl_players[["player", "nation", "club", "minutes", "goals", "assists", "position"]]
    .assign(ga=lambda d: d["goals"] + d["assists"])
    .rename(columns={"player": "Player", "nation": "Nation", "club": "Club",
                     "minutes": "Min", "goals": "G", "assists": "A",
                     "ga": "G+A", "position": "Pos"})
    .sort_values("Min", ascending=False)
    .reset_index(drop=True)
)
# Apply logo/colour while Club still holds FBref names (footy knows them)
player_detail["colour"]      = player_detail["Club"].apply(_colour)
player_detail["logo"]        = player_detail["Club"].apply(_logo)
# Then remap to EPL display names
player_detail["Club"]        = player_detail["Club"].map(club_name_map).fillna(player_detail["Club"])
player_detail["nation_flag"] = player_detail["Nation"].map(NATION_FLAGS).fillna("🌐")
player_detail["Player_flag"] = player_detail["Player"] + "  " + player_detail["nation_flag"]

# ── Players with G+A > 0 for production dropdown ─────────────────────────────
ga_players = (
    epl_players[epl_players["goals"] + epl_players["assists"] > 0]
    .copy()
    .rename(columns={"player": "Player", "nation": "Nation", "club": "Club",
                     "minutes": "Min", "goals": "G", "assists": "A",
                     "position": "Pos"})
)
ga_players["G+A"]        = ga_players["G"] + ga_players["A"]
ga_players["nineties"]   = ga_players["Min"] / 90
ga_players["ga_per90"]   = (ga_players["G+A"] / ga_players["nineties"]).round(2)
ga_players["colour"]     = ga_players["Club"].apply(_colour)
ga_players["logo"]       = ga_players["Club"].apply(_logo)
ga_players["Club"]       = ga_players["Club"].map(club_name_map).fillna(ga_players["Club"])
ga_players["nation_flag"]  = ga_players["Nation"].map(NATION_FLAGS).fillna("🌐")
ga_players["Player_flag"]  = ga_players["Player"] + "  " + ga_players["nation_flag"]

# ── Pre-compute linear regression for scatter annotation ─────────────────────
_reg_data = merged[merged["position"].notna() & (merged["wc_minutes"] >= 0)].copy()
_reg_data["position"] = _reg_data["position"].astype(int)
_slope, _intercept, _r, _p, _ = scipy_stats.linregress(
    _reg_data["position"], _reg_data["wc_minutes"]
)
_r2 = _r ** 2
_x = np.array([1, 20])
_reg_line = pd.DataFrame({"position": _x,
                          "wc_minutes": _slope * _x + _intercept})

In [3]:
#| label: chart-epl-club-minutes
#| fig-cap: Total World Cup minutes by EPL club. Club logo centred in each bar; minutes shown in parentheses at the bar end.

top_clubs = merged[merged["wc_minutes"] > 0].nlargest(20, "wc_minutes").copy()
top_clubs["mid_x"]     = top_clubs["wc_minutes"] / 2
top_clubs["min_label"] = top_clubs["wc_minutes"].apply(lambda x: f"({x:,})")
sort_field = alt.EncodingSortField(field="wc_minutes", order="descending")

base = alt.Chart(top_clubs)

bars = base.mark_bar().encode(
    x=alt.X("wc_minutes:Q",
            axis=alt.Axis(title="Total World Cup minutes", format=",d")),
    y=alt.Y("display_name:N", sort=sort_field, axis=alt.Axis(title=None)),
    color=alt.Color("colour:N", scale=None, legend=None),
    tooltip=[
        alt.Tooltip("display_name:N", title="Club"),
        alt.Tooltip("wc_minutes:Q",   title="WC minutes",  format=",d"),
        alt.Tooltip("player_count:Q", title="Players"),
        alt.Tooltip("wc_goals:Q",     title="Goals"),
        alt.Tooltip("wc_assists:Q",   title="Assists"),
        alt.Tooltip("ga_per90:Q",     title="G+A per 90",  format=".2f"),
        alt.Tooltip("position:Q",     title="EPL position"),
    ],
)

# Logo centred in bar (only when bar is wide enough to fit it)
center_logos = base.mark_image(width=26, height=26).encode(
    x=alt.X("mid_x:Q"),
    y=alt.Y("display_name:N", sort=sort_field),
    url="logo:N",
    opacity=alt.condition(
        alt.datum.wc_minutes >= 600,
        alt.value(1), alt.value(0)
    ),
    tooltip=[
        alt.Tooltip("display_name:N", title="Club"),
        alt.Tooltip("wc_minutes:Q",   title="WC minutes", format=",d"),
    ],
)

# Minutes in parentheses at the bar end
end_labels = base.mark_text(align="left", dx=5, fontSize=10,
                             color="#333333").encode(
    x=alt.X("wc_minutes:Q"),
    y=alt.Y("display_name:N", sort=sort_field),
    text="min_label:N",
)

(bars + center_logos + end_labels).properties(
    width=620, height=520,
    title="2026 World Cup: Minutes by EPL Club"
).interactive()

alt.LayerChart(...)

In [4]:
#| label: chart-nation-stacked
#| fig-cap: World Cup minutes by EPL club, stacked by national team. Flag emoji centred in each segment; legend shows flags only (no colour blocks).

flag_order = (
    [NATION_FLAGS.get(n, "🌐") + " " + n for n in
     sorted(top_nations,
            key=lambda n: -epl_players[epl_players["nation"]==n]["minutes"].sum())]
    + ["🌍 Other"]
)

# Stacked bars — Vega-Lite handles the stacking; segments ordered largest-first
nation_bars = alt.Chart(nbc_agg).mark_bar().encode(
    x=alt.X("minutes:Q",
            axis=alt.Axis(title="World Cup minutes", format=",d")),
    y=alt.Y("display_name:N", sort=top_club_display, axis=alt.Axis(title=None)),
    color=alt.Color(
        "nation_flag:N",
        sort=flag_order,
        scale=alt.Scale(scheme="tableau20"),
        legend=alt.Legend(
            title=None,
            orient="right",
            symbolSize=80,
            labelFontSize=13,
        ),
    ),
    order=alt.Order("minutes:Q", sort="descending"),
    tooltip=[
        alt.Tooltip("display_name:N", title="Club"),
        alt.Tooltip("nation_flag:N",  title="Nation"),
        alt.Tooltip("minutes:Q",      title="Minutes", format=",d"),
        alt.Tooltip("player_count:Q", title="Players"),
    ],
)

# Flag emoji centred in each segment using pre-computed x_mid positions
flag_text = alt.Chart(nbc_sorted).mark_text(fontSize=13, baseline="middle").encode(
    x=alt.X("x_mid:Q"),
    y=alt.Y("display_name:N", sort=top_club_display),
    text="flag_only:N",
    opacity=alt.condition(
        alt.datum.minutes >= 200,
        alt.value(1), alt.value(0)
    ),
    tooltip=[
        alt.Tooltip("display_name:N", title="Club"),
        alt.Tooltip("nation_flag:N",  title="Nation"),
        alt.Tooltip("minutes:Q",      title="Minutes", format=",d"),
    ],
)

(nation_bars + flag_text).properties(
    width=680, height=480,
    title="2026 World Cup: National Team Breakdown by EPL Club"
).interactive()

alt.LayerChart(...)

In [5]:
#| label: chart-players
#| fig-cap: EPL players at the 2026 World Cup, sorted by minutes. Bar colour and logo are the club's.

top60 = player_detail.head(60).copy()
p_sort = alt.EncodingSortField(field="Min", order="descending")

p_bars = (
    alt.Chart(top60)
    .mark_bar()
    .encode(
        x=alt.X("Min:Q",
                axis=alt.Axis(title="World Cup minutes", format=",d")),
        y=alt.Y("Player_flag:N", sort=p_sort,
                axis=alt.Axis(title=None, labelLimit=200)),
        color=alt.Color("colour:N", scale=None, legend=None),
        tooltip=[
            alt.Tooltip("Player:N",  title="Player"),
            alt.Tooltip("Nation:N",  title="Nation"),
            alt.Tooltip("Club:N",    title="Club"),
            alt.Tooltip("Pos:N",     title="Position"),
            alt.Tooltip("Min:Q",     title="Minutes", format=",d"),
            alt.Tooltip("G:Q",       title="Goals"),
            alt.Tooltip("A:Q",       title="Assists"),
            alt.Tooltip("G+A:Q",     title="G+A"),
        ],
    )
)

p_logos = (
    alt.Chart(top60)
    .mark_image(width=18, height=18)
    .encode(
        x=alt.value(9),
        y=alt.Y("Player_flag:N", sort=p_sort),
        url="logo:N",
        tooltip=[
            alt.Tooltip("Player:N", title="Player"),
            alt.Tooltip("Club:N",   title="Club"),
            alt.Tooltip("Nation:N", title="Nation"),
        ],
    )
)

(p_bars + p_logos).properties(
    width=640, height=980,
    title="EPL Players at the 2026 World Cup (top 60 by minutes)"
).interactive()

alt.LayerChart(...)

In [6]:
#| label: chart-scatter-position-minutes
#| fig-cap: EPL final position vs. World Cup minutes per club. Each logo is one club.

scatter_data = _reg_data.copy()

logos_scatter = (
    alt.Chart(scatter_data)
    .mark_image(width=32, height=32)
    .encode(
        x=alt.X("position:Q",
                scale=alt.Scale(domain=[1, 20]),
                axis=alt.Axis(title="2025/26 EPL final position (1 = champion)")),
        y=alt.Y("wc_minutes:Q",
                axis=alt.Axis(title="World Cup minutes", format=",d")),
        url="logo:N",
        tooltip=[
            alt.Tooltip("display_name:N", title="Club"),
            alt.Tooltip("position:Q",     title="EPL position"),
            alt.Tooltip("wc_minutes:Q",   title="WC minutes",  format=",d"),
            alt.Tooltip("player_count:Q", title="Players"),
            alt.Tooltip("ga_per90:Q",     title="G+A per 90",  format=".2f"),
        ],
    )
)

trend = (
    alt.Chart(_reg_line)
    .mark_line(color="#CC0000", strokeDash=[5, 3], opacity=0.7)
    .encode(x=alt.X("position:Q"), y=alt.Y("wc_minutes:Q"))
)

_ann_y = int(scatter_data["wc_minutes"].max() * 0.94)
annotation = (
    alt.Chart(pd.DataFrame({
        "x": [10.5], "y": [_ann_y],
        "t": [f"OLS: slope = {_slope:.0f} min/place  |  R² = {_r2:.2f}  |  p = {_p:.3f}"],
    }))
    .mark_text(fontSize=10, color="#CC0000", align="center")
    .encode(x=alt.X("x:Q"), y=alt.Y("y:Q"), text="t:N")
)

(logos_scatter + trend + annotation).properties(
    width=620, height=440,
    title="EPL Position vs. World Cup Minutes (2025/26)"
).interactive()

alt.LayerChart(...)

In [7]:
#| label: chart-epl-club-ga
#| fig-cap: G+A per 90 for individual players, by EPL club. Select a club from the dropdown.

club_opts = ["Arsenal"] + sorted(
    c for c in ga_players["Club"].dropna().unique() if c != "Arsenal"
)
club_sel = alt.selection_point(
    fields=["Club"],
    bind=alt.binding_select(options=club_opts, name="Club: "),
    value="Arsenal",
)

ga_p_sort = alt.EncodingSortField(field="ga_per90", order="descending")

ga_p_bars = (
    alt.Chart(ga_players)
    .mark_bar()
    .encode(
        x=alt.X("ga_per90:Q",
                axis=alt.Axis(title="G+A per 90 min")),
        y=alt.Y("Player_flag:N", sort=ga_p_sort,
                axis=alt.Axis(title=None, labelLimit=200)),
        color=alt.Color("colour:N", scale=None, legend=None),
        tooltip=[
            alt.Tooltip("Player:N",   title="Player"),
            alt.Tooltip("Nation:N",   title="Nation"),
            alt.Tooltip("ga_per90:Q", title="G+A per 90",  format=".2f"),
            alt.Tooltip("G+A:Q",      title="G+A"),
            alt.Tooltip("G:Q",        title="Goals"),
            alt.Tooltip("A:Q",        title="Assists"),
            alt.Tooltip("Min:Q",      title="Minutes",      format=",d"),
        ],
    )
    .add_params(club_sel)
    .transform_filter(club_sel)
)

ga_p_logos = (
    alt.Chart(ga_players)
    .mark_image(width=18, height=18)
    .encode(
        x=alt.value(9),
        y=alt.Y("Player_flag:N", sort=ga_p_sort),
        url="logo:N",
        tooltip=[alt.Tooltip("Player:N", title="Player"),
                 alt.Tooltip("Club:N",   title="Club")],
    )
    .add_params(club_sel)
    .transform_filter(club_sel)
)

(ga_p_bars + ga_p_logos).properties(
    width=500, height=300,
    title="2026 World Cup: G+A per 90 by Player (select club)"
).interactive()

alt.LayerChart(...)